# Fine-tuning Data Preparation & QLoRA

Prepare training data to fine-tune **Llama-3.2-3B** with QLoRA for product price prediction.
The actual GPU training happens on Colab

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer
from tqdm import tqdm

load_dotenv(override=True)

random.seed(42)
np.random.seed(42)
print("Imports OK")

## Load the items dataset

In [ ]:
ds = load_dataset("ed-donner/items_lite")
print(ds)

train_items = list(ds["train"])
val_items   = list(ds["validation"])
test_items  = list(ds["test"])

print(f"\nTrain: {len(train_items):,} | Val: {len(val_items):,} | Test: {len(test_items):,}")
print("\nSample item keys:", list(train_items[0].keys()))
print("\nSample summary prefix:\n", train_items[0]["summary"][:200])

## Build Llama chat-template prompt/completion pairs

Fine-tuning requires the model to see the **exact format** it will receive at inference time.
We use the Llama-3 instruct chat template.

In [ ]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"
PREFIX     = "Price is $"
QUESTION   = "What does this cost to the nearest dollar?"


def make_llama_pair(item: dict) -> dict:
    """Return a dict with 'text' field containing the full prompt+completion."""
    description = item["summary"]
    price_str = str(round(item["price"]))

    # Llama-3 instruct format
    text = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n"
        "You are an expert in estimating product prices based on descriptions.\n"
        "Reply with ONLY the price in dollars as a whole number, no $ sign.<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n"
        f"{QUESTION}\n\n{description}<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n"
        f"{price_str}<|eot_id|>"
    )
    return {"text": text, "price": item["price"], "category": item.get("category", "")}


# Build pairs for all splits
train_pairs = [make_llama_pair(item) for item in train_items]
val_pairs   = [make_llama_pair(item) for item in val_items]
test_pairs  = [make_llama_pair(item) for item in test_items]

print("Sample formatted text:")
print(train_pairs[0]["text"][:500])

## Tokenise & analyse token-length distribution

In [ ]:
# NOTE: accessing the Llama tokenizer requires a HuggingFace token + model access agreement.
# If you don't have access yet, we'll use a proxy tokenizer (GPT-2) for the length distribution.

HF_TOKEN = os.getenv("HF_TOKEN")

try:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
    print(f"Loaded tokenizer: {BASE_MODEL}")
except Exception as e:
    print(f"Could not load Llama tokenizer ({e}).\nFalling back to GPT-2 for length analysis.")
    tokenizer = AutoTokenizer.from_pretrained("gpt2")

tokenizer.pad_token = tokenizer.eos_token

# Measure token lengths
lengths = [len(tokenizer.encode(pair["text"])) for pair in tqdm(train_pairs[:2000], desc="Tokenising")]

print(f"\nToken lengths — min: {min(lengths)}  max: {max(lengths)}  mean: {np.mean(lengths):.0f}  p95: {np.percentile(lengths, 95):.0f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(lengths, bins=40, color="steelblue", edgecolor="white")
ax.axvline(110, color="red", linestyle="--", label="110-token cut-off")
ax.set_xlabel("Token length"); ax.set_ylabel("Count")
ax.set_title("Token-length distribution (first 2000 training examples)")
ax.legend()
plt.tight_layout()
plt.show()

## Filter to ≤ 110 tokens

Keeping only examples that fit comfortably in the LLM context reduces training cost and
avoids truncation artefacts.

In [ ]:
MAX_TOKENS = 300

def within_limit(pair: dict) -> bool:
    return len(tokenizer.encode(pair["text"])) <= MAX_TOKENS

filtered_train = [p for p in tqdm(train_pairs, desc="Filtering train") if within_limit(p)]
filtered_val   = [p for p in tqdm(val_pairs,   desc="Filtering val")   if within_limit(p)]
filtered_test  = [p for p in tqdm(test_pairs,  desc="Filtering test")  if within_limit(p)]

print(f"After filtering (≤{MAX_TOKENS} tokens):")
print(f"  Train: {len(filtered_train):,} ({len(filtered_train)/len(train_pairs)*100:.1f}% retained)")
print(f"  Val:   {len(filtered_val):,}")
print(f"  Test:  {len(filtered_test):,}")

## Save locally & (optional) push to HuggingFace Hub

In [ ]:
# Build HuggingFace DatasetDict
prepared_ds = DatasetDict({
    "train":      Dataset.from_list(filtered_train),
    "validation": Dataset.from_list(filtered_val),
    "test":       Dataset.from_list(filtered_test),
})

print(prepared_ds)
print("\nSample training example:")
print(prepared_ds["train"][0]["text"])

# Save to disk
prepared_ds.save_to_disk("./items_fine_tune_ready")
print("\nDataset saved locally to ./items_fine_tune_ready")

# Optional: push to your HuggingFace Hub repo
# HF_REPO = "<your-hf-username>/items-llama-finetune"
# prepared_ds.push_to_hub(HF_REPO, token=HF_TOKEN)
# print(f"Pushed to HuggingFace Hub: {HF_REPO}")

## QLoRA Training Plan (for Colab)

The following code is **not run here** — it runs on a GPU in Google Colab.
This cell documents the key QLoRA hyperparameters used for the fine-tuning run.

In [ ]:
# ── QLoRA training config (reference only — run on Colab with GPU) ─────────────

QLORA_CONFIG = """
# --- QLoRA Hyperparameters ---
# Base model: meta-llama/Llama-3.2-3B
# Dataset:    <your-hf-username>/items-llama-finetune
#
# BitsAndBytes quantisation:
#   load_in_4bit=True, bnb_4bit_quant_type='nf4', compute_dtype=float16
#
# LoRA config:
#   r=32, lora_alpha=64, target_modules=['q_proj','v_proj','k_proj','o_proj'],
#   lora_dropout=0.05, bias='none', task_type='CAUSAL_LM'
#
# TrainingArguments:
#   num_train_epochs=3, per_device_train_batch_size=4,
#   gradient_accumulation_steps=4, warmup_steps=50,
#   learning_rate=2e-4, fp16=True, logging_steps=25,
#   save_strategy='epoch', output_dir='./lora-llama-pricer'
"""

print(QLORA_CONFIG)

## Evaluate the fine-tuned model (optional)

For a fine-tuned model on HuggingFace Hub, load it and run evaluation.
Otherwise this section shows how evaluation would work.

In [ ]:
# Uncomment and fill in your fine-tuned model repo when available
#
# from transformers import AutoModelForCausalLM, pipeline
# from peft import PeftModel
# import torch
#
# FINETUNED_MODEL = "<your-hf-username>/llama-pricer-lora"  # replace with your repo
#
# base   = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map='auto', torch_dtype=torch.float16)
# model  = PeftModel.from_pretrained(base, FINETUNED_MODEL)
# pipe   = pipeline('text-generation', model=model, tokenizer=tokenizer, max_new_tokens=5)
#
# def predict_with_finetune(description: str) -> float:
#     prompt = (
#         '<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n'
#         f'What does this cost to the nearest dollar?\n\n{description}<|eot_id|>'
#         '<|start_header_id|>assistant<|end_header_id|>\n'
#     )
#     out = pipe(prompt)[0]['generated_text']
#     try:
#         return float(out.split('<|start_header_id|>assistant<|end_header_id|>')[-1].strip())
#     except ValueError:
#         return 100.0
#
# from sklearn.metrics import mean_absolute_error
# test_subset = filtered_test[:50]
# preds = [predict_with_finetune(p['text'].split('<|start_header_id|>user<|end_header_id|>')[1]
#          .split('<|eot_id|>')[0].strip()) for p in tqdm(test_subset)]
# mae = mean_absolute_error([p['price'] for p in test_subset], preds)
# print(f'Fine-tuned Llama MAE = ${mae:.2f}')

print("Fine-tuning evaluation cell ready — fill in your model repo path to activate.")